# Lab 4: Black-box distillation

Here the teacher is used only through its text replies, the way an API teacher would be.

You will:

1. Try two prompts on the validation set and get a verdict: is this teacher worth learning from?
2. Label part of the unlabelled pool with the better prompt.
3. Have the teacher write new messages, and filter them.
4. Train two students and score them on the real test set.

This lab needs the results saved by Lab 2. Allow about 45 minutes. Run the cells in order.

In [ ]:
from kd_course.env import check_environment
from kd_course.results import require_result

report = check_environment()
print(report.summary())
assert report.ok, "Fix the problems listed above before continuing."

baseline_result = require_result("baseline_gold_616")

BATCH_SIZE = 8  # lower this if the longer prompt runs out of memory
POOL_LIMIT = 3000  # how many pool messages to label; more is better and slower

In [ ]:
from kd_course.data import load_banking77, make_splits

bundle = load_banking77()
splits = make_splits(bundle.train, bundle.test)
LABELS = bundle.label_names
N = len(LABELS)

## 1. The teacher, as a black box

`LocalTeacher` wraps the local model so that the only thing the rest of the code can do is send
text and read text back. To use an API teacher, replace the last line with
`teacher = ApiTeacher.from_env()` after reading lesson 4.5.

In [ ]:
from kd_course.models import TEACHER_ID, load_causal_lm
from kd_course.teachers import LocalTeacher

model, tokenizer = load_causal_lm(TEACHER_ID)
teacher = LocalTeacher(model, tokenizer, batch_size=BATCH_SIZE)

## 2. Two prompts, compared on the validation set

The plain prompt lists the 77 types. The second prompt also shows one real message for each type,
taken from the gold training set.

In [ ]:
from kd_course.metrics import accuracy
from kd_course.synthetic import pick_examples
from kd_course.teachers import classify_with_teacher

examples = pick_examples(splits.gold_train, LABELS, per_class=1)
prompts = {"plain": None, "one example per type": examples}

val_accuracy = {}
for name, prompt_examples in prompts.items():
    answers = classify_with_teacher(teacher, splits.gold_val.texts, LABELS, prompt_examples)
    val_accuracy[name] = accuracy(splits.gold_val.labels, answers)
    unusable = sum(a is None for a in answers)
    print(f"{name:<22} validation accuracy {val_accuracy[name]:.1%}, unusable replies {unusable}")

best_prompt = max(val_accuracy, key=val_accuracy.get)
best_examples = prompts[best_prompt]
print("Using:", best_prompt)

## 3. The verdict

Is the teacher, with its better prompt, clearly ahead of the baseline on the validation set?
The lab carries on whatever the answer, so you can see what happens. Lesson 4.5 says what each
verdict would mean on a real project.

In [ ]:
from kd_course.metrics import teacher_verdict

baseline_val = baseline_result["metrics"].get("val_accuracy")
print(teacher_verdict(val_accuracy[best_prompt], baseline_val, len(splits.gold_val)))

## 4. Label part of the pool

Unusable replies are dropped, and counted.

In [ ]:
from kd_course.synthetic import labelled_from_predictions

pool_texts = splits.pool.texts[:POOL_LIMIT]
pool_answers = classify_with_teacher(teacher, pool_texts, LABELS, best_examples)
pool_labelled, dropped = labelled_from_predictions(pool_texts, pool_answers)
print(f"Labelled {len(pool_labelled)} messages; dropped {dropped} unusable replies.")

# The answer key at the back of the book: the pool's true labels are stored in this course.
# On a real project you would read a sample by hand instead.
truth = splits.pool.labels[:POOL_LIMIT]
print(f"Teacher accuracy on these pool messages: {accuracy(truth, pool_answers):.1%}")

## 5. The teacher's score on the test set

With the chosen prompt. This is the ceiling for this lab.

In [ ]:
from kd_course.metrics import summarize
from kd_course.results import save_result

teacher_pred = classify_with_teacher(teacher, splits.test.texts, LABELS, best_examples)
teacher_result = summarize(splits.test.labels, teacher_pred, N)
save_result("teacher_prompted_best", teacher_result, teacher_pred)
print(teacher_result)

## 6. Have the teacher write new messages

Two real messages per type are shown as examples, from the gold training set only. Repeats are
removed, along with anything that matches a test or validation message. Then the round-trip check
drops messages the teacher itself assigns to a different type.

In [ ]:
from kd_course.synthetic import dedupe, generate_examples, round_trip_filter

seeds = {label: [] for label in range(N)}
for text, label in zip(splits.gold_train.texts, splits.gold_train.labels, strict=True):
    if len(seeds[label]) < 2:
        seeds[label].append(text)

generated = generate_examples(teacher, LABELS, n_per_class=8, seed_examples=seeds)
written = len(generated)
protected = splits.test.texts + splits.gold_val.texts
generated, repeats = dedupe(generated, against=protected)
generated, off_type = round_trip_filter(teacher, generated, LABELS, best_examples)
print(
    f"Written {written}; removed {repeats} repeats and {off_type} off-type; kept {len(generated)}."
)
print("Three examples:", generated.texts[:3])

In [ ]:
import gc

import torch

del teacher, model
gc.collect()
torch.cuda.empty_cache()

## 7. Train two students

In [ ]:
from kd_course.classifier import STUDENT_ENCODER_ID, predict, train_classifier
from kd_course.data import LabelledSet


def train_and_score(name, train_set):
    student = train_classifier(
        STUDENT_ENCODER_ID, train_set, splits.gold_val, num_labels=N, epochs=5, batch_size=32
    )
    predictions = predict(student, splits.test.texts)
    scores = summarize(splits.test.labels, predictions, N)
    save_result(name, scores, predictions)
    del student
    gc.collect()
    torch.cuda.empty_cache()
    return scores, predictions


labelled = LabelledSet(
    texts=splits.gold_train.texts + pool_labelled.texts,
    labels=splits.gold_train.labels + pool_labelled.labels,
)
labelled_scores, labelled_pred = train_and_score("blackbox_labelled_pool", labelled)

with_generated = LabelledSet(
    texts=labelled.texts + generated.texts, labels=labelled.labels + generated.labels
)
generated_scores, generated_pred = train_and_score("blackbox_plus_generated", with_generated)

## 8. Compare

In [ ]:
from kd_course.metrics import gap_closed, paired_bootstrap
from kd_course.results import format_results, load_results

print(format_results(load_results()))
print()

base_acc = baseline_result["metrics"]["accuracy"]
gap = gap_closed(base_acc, labelled_scores["accuracy"], teacher_result["accuracy"])
print("Gap closed:", "no gap to close" if gap is None else f"{gap:.0%}")


def report(title, a, b):
    r = paired_bootstrap(accuracy, splits.test.labels, a, b)
    print(
        f"{title}: {r.difference * 100:+.1f} points "
        f"(95% interval {r.low * 100:+.1f} to {r.high * 100:+.1f})"
    )


report("Labelled pool minus baseline      ", baseline_result["predictions"], labelled_pred)
report("Generated added minus labelled pool", labelled_pred, generated_pred)

## 9. What did you see?

Double-click this cell and write your answers.

**a. What was your verdict in step 3? Did the students' results match what lesson 4.5 said to expect for that verdict?**

**b. Did the example prompt beat the plain prompt on validation? Was the difference larger than the margin of error for 308 messages?**

**c. The teacher's accuracy on the pool and the student's accuracy on the test set are both printed. Is the student above or below its teacher's labels? Use lesson 4.3 to explain why.**

**d. Did generated messages help? Give the interval. Why might they matter more if you had only 300 pool messages instead of 3,000?**

**e. If you ran Lab 3: compare `distilled_top_answers` and `distilled_probabilities_T2` with this lab's students. Which approach would you use for this task, and what would change your mind?**